In [1]:
from pathlib import Path

import scanpy as sc
import pandas as pd
import numpy as np

DATA = Path("../data/processed")
RESULTS = Path("../results/phase0")

RESULTS.mkdir(parents=True, exist_ok=True)


In [2]:
k562 = sc.read_h5ad(
    DATA / "replogle22k562.h5ad",
    backed="r"
)

rpe1 = sc.read_h5ad(
    DATA / "replogle22rpe1.h5ad",
    backed="r"
)

print("K562:", k562.shape)
print("RPE1:", rpe1.shape)

K562: (308646, 8563)
RPE1: (240774, 8749)


In [3]:
print("K562 columns:")
print(k562.obs.columns.tolist())

print("\nRPE1 columns:")
print(rpe1.obs.columns.tolist())

print("\nK562 perturbations:")
print(k562.obs["perturbation"].value_counts().head(10))

print("\nRPE1 perturbations:")
print(rpe1.obs["perturbation"].value_counts().head(10))

K562 columns:
['perturbation']

RPE1 columns:
['perturbation']

K562 perturbations:
perturbation
control    10691
RPL3        1996
NCBP2        992
KIF11        974
SLC39A9      752
DONSON       745
NUP205       735
GAB2         674
BUB1         648
SNX15        637
Name: count, dtype: int64

RPE1 perturbations:
perturbation
control    11485
TFAM        3580
SLC1A5      1962
GFM1        1699
MRPL36      1686
TARDBP      1675
PPP6C       1581
MRPL35      1491
NBPF12      1159
CCDC6       1086
Name: count, dtype: int64


In [4]:
def summarize_dataset(adata, name):
    counts = adata.obs["perturbation"].value_counts()

    perturbation_counts = counts.drop(
        labels="control",
        errors="ignore"
    )

    print(name)
    print("-" * 40)
    print(f"Cells: {adata.n_obs:,}")
    print(f"Measured genes: {adata.n_vars:,}")
    print(f"Controls: {counts.get('control', 0):,}")
    print(f"Perturbations: {len(perturbation_counts):,}")
    print(
        f"Median cells per perturbation: "
        f"{perturbation_counts.median():.0f}"
    )
    print()


summarize_dataset(k562, "K562")
summarize_dataset(rpe1, "RPE1")

K562
----------------------------------------
Cells: 308,646
Measured genes: 8,563
Controls: 10,691
Perturbations: 1,971
Median cells per perturbation: 125

RPE1
----------------------------------------
Cells: 240,774
Measured genes: 8,749
Controls: 11,485
Perturbations: 2,016
Median cells per perturbation: 82



In [5]:
k562_perts = (
    set(k562.obs["perturbation"].unique())
    - {"control"}
)

rpe1_perts = (
    set(rpe1.obs["perturbation"].unique())
    - {"control"}
)

shared_perts = sorted(
    k562_perts & rpe1_perts
)

print("K562 perturbations:", len(k562_perts))
print("RPE1 perturbations:", len(rpe1_perts))
print("Shared perturbations:", len(shared_perts))

print("\nExamples:")
print(shared_perts[:20])

K562 perturbations: 1971
RPE1 perturbations: 2016
Shared perturbations: 1676

Examples:
['AAAS', 'AARS', 'AARS2', 'AASDHPPT', 'ABCB7', 'ABCE1', 'ABCF1', 'ABCG1', 'ABHD11', 'ABHD17A', 'ACD', 'ACIN1', 'ACOT12', 'ACTB', 'ACTR1A', 'ACTR1B', 'ACTR2', 'ACTR6', 'ACTR8', 'ADAT2']


In [6]:
k562_genes = set(k562.var_names)
rpe1_genes = set(rpe1.var_names)

shared_genes = sorted(
    k562_genes & rpe1_genes
)

print("K562 genes:", len(k562_genes))
print("RPE1 genes:", len(rpe1_genes))
print("Shared genes:", len(shared_genes))

K562 genes: 8563
RPE1 genes: 8749
Shared genes: 7226


In [7]:
audit = pd.DataFrame(
    {
        "dataset": ["K562", "RPE1"],
        "n_cells": [
            k562.n_obs,
            rpe1.n_obs
        ],
        "n_genes": [
            k562.n_vars,
            rpe1.n_vars
        ],
        "n_perturbations": [
            len(k562_perts),
            len(rpe1_perts)
        ],
        "n_shared_perturbations": [
            len(shared_perts),
            len(shared_perts)
        ],
        "n_shared_genes": [
            len(shared_genes),
            len(shared_genes)
        ]
    }
)

audit

,dataset,n_cells,n_genes,n_perturbations,n_shared_perturbations,n_shared_genes
0,K562,308646,8563,1971,1676,7226
1,RPE1,240774,8749,2016,1676,7226


In [8]:
audit.to_csv(
    RESULTS / "dataset_audit.csv",
    index=False
)